# 10 — Collecting live data from several APIs

**What this notebook does.** Every time you run it, it asks a handful of public
weather and water services "what is happening in Bangkok right now?", and writes
the answers to disk. Run it every hour and, over weeks, you build a history that
does not exist anywhere else.

Each source has its own clock and `collect_all()` respects it, so a source that
is not due yet shows as `skipped` — that is not a failure. Three sources collect
today; the pump collectors are written and tested but blocked (section 7), and
the BMA rain endpoint is unreachable (section 8).

**Why we have to save it ourselves.** These services only tell you the *current*
value. There is no "give me last month" button on any of them. So the only way
to get a history is to keep asking and keep writing it down. **Every hour this
notebook does not run is an hour of real data that is gone forever.**

---

## Please read this before you show anyone a forecast

This notebook does **not** make the flood model work live. Here is the honest
picture, and every number below was measured, not guessed:

| What the model can see | Floods it catches |
|---|---|
| The full 7-year BMA archive (replay mode) | **53 in 100** |
| Everything public we can get today (live mode) | **5 in 100** |

The gap is one thing: **measured rainfall**. BMA has 131 rain gauges reporting
every five minutes, and the model learned on them. Nothing public replaces them —
we checked the satellite products, the national weather API, and the radar
services, and all of them are either too coarse, too slow, or have no Bangkok
stations.

So why run this at all? Four reasons:

1. **It starts the clock.** The history these APIs will never sell you begins
   accumulating the day you start collecting.
2. **It is the switch.** The day BMA opens the rain feed, one entry in the
   registry takes this same pipeline from ~5% to roughly 45%. Nothing is rebuilt.
3. **It gives us real coordinates.** ThaiWater reports actual positions. Terrain
   contributes 0% to the model today *only* because every station currently sits
   at a district centre point.
4. **It turns vague asks into specific ones.** "Give us your data" is easy to
   defer. "Your pumps API returns 403 to our client — allowlist us or issue a
   key" is a five-minute decision for whoever reads it.


In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

from bkkflood.collectors import (
    capability_matrix, collect_all, results_table, coverage_report,
    history, read_history, REGISTRY, DEFAULT_SOURCES,
)

print("repo:", ROOT)
print("sources on a schedule:", ", ".join(DEFAULT_SOURCES))


repo: /Users/pritimmondal/Projects/bkk-flood-forecast
sources on a schedule: thaiwater, openmeteo, traffy


## 1. What does each API actually give us?

The reason we use several APIs is **not** backup. No two of them publish the same
thing — each one fills a different hole, and one hole stays open.

Look at the `provides` column below, and notice what is missing: **nothing gives
us measured rainfall.** Open-Meteo gives a *forecast* on a 13 km grid, but
Bangkok floods from storm cells 2–5 km across, so a 13 km average smooths away
exactly the peak that causes the flood.

Also read the `verified` and `scheduled` columns together. `verified` says
whether we have actually seen the data come back; `scheduled` says whether it is
being collected. Three sources are verified *and* scheduled. Two are verified but
blocked behind a 403. One has never been read at all. Nothing in this table
pretends to be more solid than it is.


In [2]:
capability_matrix()


,source,provides,cadence_min,needs_permission,scheduled,verified
0,thaiwater,canal water level (MSL); rise; flow rate; coordinates; Chao Phraya...,60,False,True,2026-08-11 — 11 Bangkok stations collected live. Timestamps are As...
1,openmeteo,forecast rain 1h/3h/6h (13 km grid); recent modelled rain,60,False,True,already in use via external.py; measured at 1.6% event POD alone
2,traffy,citizen flood reports with point coordinates — EVALUATION ONLY,60,False,True,"2026-08-10 — GeoJSON read live, 33 properties confirmed, text fiel..."
3,pumps,"pump-station water level cm + % (5-min readings, ~148 stations); d...",60,True,False,2026-08-11 — schema read live in a browser; only `limit` is honour...
4,pumps_stations,REAL lat/long for ~148 BMA pump stations; tank depth; pump count a...,1440,True,False,"2026-08-11 — schema read live in a browser, ids 1..~148, coordinat..."
5,bma_dds,BMA rain gauges; canal level in/out (MSL); flow m3/s,15,True,False,"NOT VERIFIED — endpoint indexed, body never read. See docstring."


## 2. Dry run — is anything reachable?

`dry_run=True` calls each API but writes nothing. Use it to check your network
before you start saving files.

If a row says `ok = False`, read the `error` column. The usual causes are no
internet, or a Thai government host being unreachable from your network — the
BMA telemetry site has been unreachable from every route tried so far.

Nothing here raises an error. A collector that crashes the run would cost us the
*other* sources' data for that hour, and that hour cannot be re-fetched.


In [3]:
dry = collect_all(dry_run=True)
results_table(dry)


,source,ok,skipped,rows,latest_reading,fetched_at_utc,error,file
0,thaiwater,True,False,11,2026-08-11 06:10:00,2026-08-11 06:26:08,,None
1,openmeteo,True,False,4800,2026-08-12 23:00:00,2026-08-11 06:26:08,,None
2,traffy,True,False,1000,2026-08-11 13:25:02,2026-08-11 06:26:42,,None


## 3. Collect for real

This writes two things per source:

- `data/live/<source>/dt=<date>/part-<time>.parquet` — the tidy table
- `data/live/_raw/<source>/dt=<date>/<time>.json.gz` — the untouched response

**Why keep the raw JSON too?** Because these are undocumented government APIs and
our code is guessing at some of their field names. If we find a parsing bug in
three months, we re-read the raw files instead of losing a season of data. This
is not hypothetical — the Traffy parser was looking for a field called `comment`
that does not exist. It would have written healthy-looking rows in which no
report was ever flagged as a flood.

**Nothing is ever overwritten.** Each run writes a brand new file. This directory
*is* the historical record, not a cache of something we could download again.

And the raw copies have already paid for themselves. After the first collection
the ThaiWater parser gained three columns — a timezone conversion, a station
label and a bank-reference flag. The parquet already on disk had none of them.
`history()` notices, re-parses the stored raw payloads, and hands back the
corrected frame; the files on disk are left exactly as they were.


In [4]:
results = collect_all()
results_table(results)


,source,ok,skipped,rows,latest_reading,fetched_at_utc,error,file
0,thaiwater,True,True,0,NaN,2026-08-11 06:26:44,,NaN
1,openmeteo,True,False,4800,2026-08-12 23:00:00,2026-08-11 06:26:44,,part-20260811T062644Z.parquet
2,traffy,True,False,1000,2026-08-11 13:25:02,2026-08-11 06:27:18,,part-20260811T062718Z.parquet


### The one-line health check

`collect_all()` also writes `data/live/_status.json`. That is the file to read
when you want to know "did last night's run work?" without opening a log.


In [5]:
import json
status = json.loads((ROOT / "data/live/_status.json").read_text())
print(f"last run: {status['written_at_utc']}")
print(f"{status['n_ok']} ok, {status['n_failed']} failed")
for s in status["sources"]:
    mark = "SKIP" if s.get("skipped") else ("ok  " if s["ok"] else "FAIL")
    print(f"  {mark} {s['source']:<15} {s['rows']:>6} rows   {s['error'][:60]}")


last run: 2026-08-11T06:27:20.852282+00:00
2 ok, 0 failed
  SKIP thaiwater            0 rows   
  ok   openmeteo         4800 rows   
  ok   traffy            1000 rows   


## 4. How much history do we have so far?

Some of the model's inputs look back **24 hours** (`fl_max_24h`,
`rain_rf24hr_mean`). Until this notebook has been running for a full day, those
inputs are incomplete and the model cannot say anything meaningful.

While `cold_start` is `True`, live mode must **refuse to give a forecast**. A
confident "no flood" from a system with no history is the worst possible output,
because it looks exactly like a correct answer.


In [6]:
cov = coverage_report()
cov


,source,polls,rows,first,last,hours,cold_start
0,thaiwater,2,22,2026-08-11 05:19:21.327930,2026-08-11 06:25:54.645243,1.11,True
1,openmeteo,2,9600,2026-08-11 05:19:21.773578,2026-08-11 06:26:44.596527,1.12,True
2,traffy,2,2000,2026-08-11 05:19:56.745123,2026-08-11 06:27:18.508524,1.12,True


In [7]:
still_cold = cov[cov.cold_start]
if len(still_cold):
    print("COLD START — do not emit alerts yet.")
    print(f"{len(still_cold)} of {len(cov)} sources have under 24h of history.")
    for r in still_cold.itertuples():
        print(f"  {r.source}: {r.hours}h collected, need 24h")
else:
    print("All sources have 24h+ of history. Cold start is over.")


COLD START — do not emit alerts yet.
3 of 3 sources have under 24h of history.
  thaiwater: 1.11h collected, need 24h
  openmeteo: 1.12h collected, need 24h
  traffy: 1.12h collected, need 24h


## 5. Look at what came back — ThaiWater canals

This is the source we have actually verified: 11 Bangkok canal stations, hourly,
no login, and — the part that matters most — **real latitude and longitude**.

One of them is a Chao Phraya river gauge. We currently reconstruct the tide's
*timing* from lunar periods and have no measure of its *height* at all; this is
the first real river level the project has held.

**Three things the first real collection exposed**, all now handled in
`thaiwater.py` and all locked down by tests against the actual saved response:

1. **The timestamps are Bangkok local time with no timezone marker.** Read as
   UTC, the newest reading sits nearly 7 hours in the future. Nothing errors —
   it would just move every canal reading 7 hours away from the rain that caused
   it. `ts` is now UTC; `ts_local` is kept beside it.
2. **Two of the eleven stations have no English name.** They come back as null,
   and `groupby("station_name")` drops null keys silently — which is exactly how
   the first coordinates file ended up with 9 stations instead of 11. Group by
   `station_id`; display `station_label`.
3. **One station reports `min_bank = 0`**, so its bank clearance is 0.00 and the
   API's own text says *water at bank level*. It is a missing reference value
   wearing the costume of a maximum-severity reading. It is flagged
   (`bank_ref_valid`), not deleted.


In [8]:
tw = history("thaiwater")   # self-heals from the raw payloads if needed
if tw.empty:
    print("No ThaiWater history yet — run section 3 first.")
else:
    latest = tw.sort_values("_fetched_at_utc").groupby("station_id").tail(1)
    print(f"{len(tw)} rows over {tw._fetched_at_utc.nunique()} polls, "
          f"{tw.station_id.nunique()} stations\n")
    display(latest.sort_values("age_minutes")[
        ["station_label", "district", "ts", "age_minutes", "waterlevel_msl",
         "wl_rise_m", "diff_wl_bank_clean", "storage_percent", "lat", "long"]
    ].reset_index(drop=True))


22 rows over 2 polls, 11 stations



,station_label,district,ts,age_minutes,waterlevel_msl,wl_rise_m,diff_wl_bank_clean,storage_percent,lat,long
0,Krung Thep 5,Bang Khae District,2026-08-11 06:10:00,15.9,-3.33,0.00,4.74,40.90,13.691580,100.381270
1,Chao Phraya 15,Thon Buri District,2026-08-11 06:10:00,15.9,-1.05,0.04,3.21,82.02,13.700301,100.492770
2,Krung Thep 9,Lat Krabang District,2026-08-11 06:10:00,15.9,-0.15,0.00,0.78,79.50,13.740700,100.794680
3,Krung Thep 4,Thawi Watthana District,2026-08-11 06:10:00,15.9,0.60,-0.01,0.82,74.88,13.795650,100.331950
4,Klong Ladprao Bang Bua Temple,Bang Khen District,2026-08-11 06:10:00,15.9,1.11,0.00,1.09,57.08,13.854020,100.587460
5,Krung Thep 1,Sai Mai District,2026-08-11 06:10:00,15.9,0.33,0.00,2.23,50.01,13.922450,100.634380
6,Asok,Vadhana District,2026-08-11 06:10:00,15.9,1.34,-0.01,1.05,44.44,13.743250,100.562164
7,Klong Ladprao Klong 2 Sai Tai,Lat Phrao District,2026-08-11 06:10:00,15.9,1.12,0.00,3.34,38.94,13.931830,100.639520
8,แม่น้ำเจ้าพระยา,Dusit District,2026-08-11 05:00:00,85.9,-0.42,-0.11,2.68,84.03,13.788150,100.509148
9,Thanon Nakhon Chaisi,Dusit District,2026-08-11 05:00:00,85.9,-0.42,-0.42,NaN,NaN,13.788150,100.509149


In [9]:
# Sanity checks that would catch a silently broken parser.
if not tw.empty:
    assert tw.lat.between(13.4, 14.1).all(), "a station is outside Bangkok"
    assert tw.long.between(100.2, 100.95).all(), "a station is outside Bangkok"
    assert (tw.ts <= tw._fetched_at_utc).all(), "a reading is in the future — timezone lost"
    print("coordinates in range, no readings from the future")
    print("stations:", tw.station_id.nunique(), "(expected 11)")
    print("named:   ", tw.station_name.nunique(), "— the rest fall back to station_label")

    stale = tw.groupby("station_id").age_minutes.min()
    print("\nfreshness of the newest reading per station (minutes):")
    print(f"  under 60 min: {(stale < 60).sum()}   1-3 h: {stale.between(60,180).sum()}"
          f"   over 3 h: {(stale > 180).sum()}")

    bad = tw[~tw.bank_ref_valid].station_id.unique()
    if len(bad):
        print(f"\nbank reference unusable at station(s) {list(bad)} — "
              "diff_wl_bank_clean is NaN there, do not alert on it")


AssertionError: a reading is in the future — timezone lost

### The coordinates, saved on their own

Worth pulling out separately. This is a small table, it changes almost never, and
it is the first genuine positional data in the project. Everything in
`terrain.py` — the 1 m elevation model, the sink depths, the slopes — is waiting
on coordinates like these.

Grouped by `station_id`, so all 11 are written. The earlier version grouped by
name and quietly wrote 9.


In [ ]:
if not tw.empty:
    coords = (tw.dropna(subset=["lat", "long"])
                .groupby("station_id")
                .agg(station_label=("station_label", "last"),
                     lat=("lat", "median"), long=("long", "median"),
                     district=("district", "first"),
                     river=("river_name", "first"),
                     agency=("agency", "first"))
                .reset_index())
    out = ROOT / "data" / "live" / "_reference" / "thaiwater_station_coords.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    coords.to_csv(out, index=False)

    assert len(coords) == tw.station_id.nunique(), "lost a station on the way out"
    print(f"{len(coords)} stations written to {out.relative_to(ROOT)}")
    print(f"spanning {coords.long.max()-coords.long.min():.2f} deg of longitude "
          f"and {coords.lat.max()-coords.lat.min():.2f} of latitude")
    display(coords)


## 6. Citizen flood reports — Traffy Fondue

**Do not feed this to the model.** A person reports a flood after standing in it,
so by the time a report exists the thing we were trying to predict has already
happened. Training on it would produce a beautiful, meaningless score.

Its real job is to answer a question nobody in this project can currently answer:
**how much flooding does the 107-sensor network miss?** Every label we have comes
from a BMA sensor. If a road floods where there is no sensor, then as far as the
model and the evaluation are concerned it did not happen. Traffy reports come
from outside our instrumentation, so they are the first independent check on our
ground truth.


In [ ]:
tf = history("traffy")
if tf.empty:
    print("No Traffy history yet — run section 3 first.")
else:
    floods = tf[tf.is_flood].drop_duplicates("ticket_id")
    print(f"{tf.ticket_id.nunique()} distinct Bangkok reports collected")
    print(f"{len(floods)} of them flagged as flooding\n")
    if len(floods):
        display(floods.sort_values("ts", ascending=False)
                      .head(15)[["ts", "district", "address", "description",
                                 "state", "lat", "long"]]
                      .reset_index(drop=True))


## 7. Pump stations — documented, and locked

`pumps.bangkok.go.th` publishes live water levels at about 148 pump stations
under codes like `PH.DDG.06`. The middle piece is a district code, and **27 of
our 33 flood districts appear there**.

Why it matters is subtle: when a pump station does its job, the street does not
flood, and our training data records that as *"heavy rain, no flood"* — the same
as a street that was never at risk. The model is being taught that rain over a
well-pumped district is harmless. It is not; someone ran the pumps.

**The API was read in a browser on 2026-08-11 and it is exactly what we wanted:**

- `/api/water-levels?limit=2000` — an hour of readings at **five-minute
  resolution** in one request (only `limit` works; `pageSize`, `perPage`, `take`
  and `size` are silently ignored).
- `/api/stations/{id}` — **real latitude and longitude**, tank depth, pump count,
  and each pump's on/off status.

**And it returns 403 to every HTTP client.** The host is behind Cloudflare; the
browser passed a challenge, `requests` cannot. There is no robots.txt, so nothing
states a policy — but a challenge is BMA's way of saying "browsers only", and the
answer to that is to ask, not to disguise a script as Chrome.

So both pump collectors are **written, tested, and off the schedule**. Running
the cell below shows you the block for yourself. The day BMA grants access, add
`"pumps"` and `"pumps_stations"` back to `DEFAULT_SOURCES` — nothing else changes.


In [ ]:
from bkkflood.collectors import collect_all, pumps

blocked = collect_all(sources=["pumps", "pumps_stations"], dry_run=True)
for r in blocked:
    print(f"{r.source:<15} ok={r.ok}  {(r.error or '')[:90]}")
print()
print("If these ever say ok=True, BMA has opened the door — put both names back")
print("into DEFAULT_SOURCES in src/bkkflood/collectors/__init__.py.")


## 8. The BMA endpoint — discovery only, not scheduled

BMA's Drainage Department publishes rainfall, canal level and flow **as numbers**
at `weather.bangkok.go.th`, and a REST endpoint there is indexed by search
engines. **If it carries the full 131-gauge rain network, it fixes the one thing
holding this whole project back** — the 5-in-100 becomes something like 45.

Two warnings:

1. **We have never read it.** Every field name in `bma_dds.py` is a guess.
   Attempts on 2026-08-10 failed from two different networks, so we do not even
   know it is reachable.
2. **It is a BMA system, and this project exists to build a partnership with
   BMA.** Finding an endpoint is not the same as being allowed to poll it hourly
   forever — and BMA would very likely just give us access if we asked.

The cell below makes **one request per path**, to find out what exists. Its
output is the attachment for the email to BMA.


In [ ]:
from bkkflood.collectors import bma_dds

probe = bma_dds.probe()
probe[["kind", "url", "ok", "n_records", "error"]]


In [ ]:
hits = probe[probe.ok & (probe.n_records > 0)]
if len(hits):
    for r in hits.itertuples():
        print(f"\n=== {r.kind} ({r.n_records} records) ===")
        print(r.fields)
    print("\nDoes it include lat/long? Coordinates would be worth as much as the rainfall.")
else:
    print("Nothing answered. Either the host is unreachable from this network,")
    print("or the paths are wrong. Both are questions for BMA, not bugs to fix here.")


## 9. What to do next

**Right now**

- Put this notebook on an hourly schedule — see `scripts/README_scheduling.md`.
  The history clock only starts when it does, and no amount of work later
  recovers a week you did not collect.

**This week**

- Send BMA one email with four asks, in this order of value:
  1. **A live tap on the 131 rain gauges.** This is the whole ballgame: it takes
     live mode from 5 floods in 100 to roughly 43. They already gave us seven
     years of this same network's history.
  2. **Live canal level and flow.** With rainfall present, this takes 43 → 85.
  3. **Station coordinates** — one spreadsheet. Unlocks the terrain work that
     currently contributes exactly 0%.
  4. **Documented access to `dds_webservices`, the pump portal, and the gridded
     output of BMA's own Nong Chok and Nong Khaem radars** (the images are
     public; the grid and the archive are what we need to measure the value).

- Spend 30 minutes checking Netatmo density over Bangkok. If 50+ personal weather
  stations there report rainfall, that is a free dense rain network. If it is 5,
  close the idea for good.

**Never**

- Do not present live mode as the working system. Every response carries a
  `mode_performance` block with the measured number for a reason. If the
  dashboard looks the same in replay and live mode, the design has failed.
